In [ ]:
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import resources
from resources.MLmetrics import (
    load_dual_diagnostics, plot_dual_loss_history, plot_dual_sample_errors, curve_summary_table, field_summary_table,
    display_curve_main_dashboard, display_curve_pointwise_summary, plot_curve_all_point_parity,
    display_curve_sample_error_summary, display_curve_initial_slope_summary,
    display_curve_peak_energy_summary, display_curve_zone_summary, display_curve_sample_examples,
    plot_field_diagnostics, plot_field_frame_component_trends, plot_field_frame_component_heatmaps,
    plot_field_diversity, plot_field_frame_component_collapse, plot_field_component_parity,
    display_field_sample_error_summary, display_field_node_error_summary,
    field_sample_viewer, plot_field_keyframe_strip, plot_field_sample_frame_evolution,
)
REPO = Path(resources.__file__).resolve().parents[1]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

# Inputs

Saved-run analysis only: no training, raw-data loading or HPC access. Trial 1—not the three-epoch execution test—is the default. Feature generation, masks and worked input examples are documented in `../samples/`, not repeated here.

In [ ]:
RUN_DIR = REPO / "data/MULTI/Dual/Transformer/dual-MULTI-trial1"
# For HPO: point RUN_DIR at the study's best/ subdirectory.
RESULTS_DIR = RUN_DIR / "results"
metrics = json.loads((RESULTS_DIR / "metrics.json").read_text())
history = pd.read_csv(RUN_DIR / "loss_history.csv")
BEST_EPOCH = metrics["best_epoch"]
SPLIT = metrics["evaluation_split"]

# General cross-task setup and configuration

The same saved specimens are shown for both tasks. A sample **index** is a row in the evaluation split, not its simulation ID. Every curve and field subsection displays both UT and FT. Plot/sample/frame controls sit beside their plots; there is no global field-task switch.

In [ ]:
CURVE_MODES = ("UT", "FT")
display(Markdown(f"**Run:** {RUN_DIR.name} · **split:** {SPLIT} · **best epoch:** {BEST_EPOCH} "
                 f"· **epochs logged:** {len(history)}"))
display(pd.DataFrame(metrics["split_sizes"].items(), columns=["Split", "Paired samples"]))
overview = []
for kind in ("curve", "field"):
    for mode in ("UT", "FT"):
        s = metrics["diagnostics"][kind][mode]
        overview.append({"Output": f"{mode} {kind}", "RMSE (native units)": s["rmse"],
                         "RMSE improvement over train mean (%)": 100*s[f"skill_vs_mean_{kind}_rmse"],
                         "Diversity ratio": s["collapse_ratio"]})
display(pd.DataFrame(overview).round(4))

## How to read the comparisons

Positive skill means lower RMSE than always predicting the **training mean**; negative skill means worse. Raw RMSE values in stress, force and displacement units are not directly comparable. A high global curve R² can mainly reflect the common loading shape rather than success at distinguishing disorder samples.

Curve diversity is mean predicted pointwise standard deviation / mean true standard deviation. Field diversity is the mean pointwise standard-deviation ratio; these are related but different summaries. Near zero indicates collapse; near one is not proof of accuracy. Baseline scores are the saved training-baseline scores; unavailable baseline arrays are not reconstructed from validation data.

# Curve Models

## Main diagnostics and dashboard

Inspect typical shape agreement and diversity before studying individual good or bad examples. The dashboard omits ratio-line points where true standard deviation is below one millionth of its peak (or 1e-12); dividing by near-zero variation is not informative. Saved metrics are unchanged.

In [ ]:
curves = {mode: load_dual_diagnostics(RUN_DIR, "curve", mode) for mode in CURVE_MODES}
DASHBOARD_MAX_SAMPLES = 64
for mode, diag in curves.items():
    display(Markdown(f"### {mode} · {SPLIT}"))
    display(curve_summary_table(diag))
    display_curve_main_dashboard(diag, mode=mode, max_samples=DASHBOARD_MAX_SAMPLES)

## Errors and correlations

Pointwise errors locate difficult loading regions. Correlations and pooled parity complement, but do not replace, baseline skill and specimen-level agreement.

In [ ]:
ERROR_CURVE_SAMPLES = 100
SHOW_CORRELATIONS = True
for mode, diag in curves.items():
    display(Markdown(f"### {mode}"))
    display_curve_pointwise_summary(diag, mode=mode, max_error_curves=ERROR_CURVE_SAMPLES,
                                    show_correlation=SHOW_CORRELATIONS)
    plot_curve_all_point_parity(diag)

## Sample errors

Use the error distribution and worst specimens to find tails hidden by average RMSE. Tables include the original simulation ID.

In [ ]:
TOP_N = 5
for mode, diag in curves.items():
    display(Markdown(f"### {mode}"))
    display_curve_sample_error_summary(diag, bins=40, ncols=3, top_n=TOP_N)

## Initial slope, peak and energy agreements

The initial-slope window is a diagnostic choice, not a new definition of material stiffness. Peak height, peak location and area agreements assess specimen-to-specimen behaviour.

In [ ]:
INITIAL_WINDOW_FRACTION = 0.08
for mode, diag in curves.items():
    display(Markdown(f"### {mode}"))
    display_curve_initial_slope_summary(diag, window_fraction=INITIAL_WINDOW_FRACTION, min_points=3, top_n=5)
    display_curve_peak_energy_summary(diag)

## Zone metrics

Use the existing task-specific curve zones saved with this run. Zone errors and diversity reveal local failures that whole-curve averages hide.

In [ ]:
for mode, diag in curves.items():
    display(Markdown(f"### {mode}"))
    display_curve_zone_summary(diag)

## Curve examples

The selected row is paired across UT/FT. The same random seed gives the same random rows; best/worst rankings can differ between tasks.

In [ ]:
SELECTED_SAMPLE = 0
RANDOM_SAMPLE_COUNT = 7
RANDOM_SEED = 42
for mode, diag in curves.items():
    display(Markdown(f"### {mode} · selected simulation ID {diag['sample_ids'][SELECTED_SAMPLE]}"))
    display_curve_sample_examples(diag, selected_sample=SELECTED_SAMPLE,
                                  random_count=RANDOM_SAMPLE_COUNT, random_seed=RANDOM_SEED,
                                  rank_by="sample_rmse", ncols=2)

## Curve loss monitoring

All loss axes are logarithmic. Weighted contributions sum to the joint objective; raw component losses do not when weights/scales differ. The dashed line marks the saved checkpoint, not necessarily the last training epoch. Train loss is measured with dropout active; validation is in evaluation mode.

In [ ]:
WEIGHTED_LOSSES = True
plot_dual_loss_history(history, kind="curve", best_epoch=BEST_EPOCH, weighted=WEIGHTED_LOSSES);

# Field Models

## Main diagnostics and dashboard

Both UT and FT are loaded once and retained for comparison. This uses more RAM than single-task review. Absent FT nodes and invalid field entries remain excluded; maps use saved reference coordinates, not a reconstructed mesh.

In [ ]:
fields = {mode: load_dual_diagnostics(RUN_DIR, "field", mode) for mode in ("UT", "FT")}
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode} · {SPLIT}"))
    display(field_summary_table(field_diag))
    plot_field_diagnostics(field_diag)

## Frame and component performances

Compare U1 and U2 and watch how errors develop over loading. Frame numbers are 1–20 in plot controls; UT and FT need not have identical physical loading coordinates.

In [ ]:
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode}"))
    plot_field_frame_component_trends(field_diag)
    plot_field_frame_component_heatmaps(field_diag)
    display(pd.DataFrame({"Displayed frame": np.arange(1, len(field_diag["frame_values"])+1),
                          "Saved loading coordinate": field_diag["frame_values"]}))

## Prediction collapse and field diversity

Standard-deviation ratios reveal whether the network distinguishes specimens. Ratios can be unstable at almost motionless nodes; inspect them alongside absolute errors and parity.

In [ ]:
PARITY_MAX_POINTS = 100000
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode}"))
    plot_field_diversity(field_diag)
    plot_field_frame_component_collapse(field_diag)
    plot_field_component_parity(field_diag, max_points_per_component=PARITY_MAX_POINTS)

## Sample errors

In [ ]:
TOP_N = 5
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode}"))
    display_field_sample_error_summary(field_diag, bins=40, ncols=3, top_n=TOP_N)

## Node errors

These are errors aggregated across specimens and frames at each retained canonical node. FT node labels retain canonical indices, so gaps in labels are expected.

In [ ]:
NODE_METRIC = "rmse"
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode}"))
    display_field_node_error_summary(field_diag, metric=NODE_METRIC, top_n=10,
                                     plot_columns=["rmse", "mae", "bias"], point_size=22)

## Interactive field viewer

**Core field-review tool:** choose specimen, frame, U1/U2, and best/worst/random ranking. Compare truth, prediction and signed error at the same locations. Truth and prediction share a colour scale; the error panel has its own scale. Colours are clipped to the 98th-percentile magnitude, so check numerical errors too.

FT defaults to points: continuous triangulation can visually fill a crack or bridge struts and is **not** a connectivity-aware solution. Continuous mode remains available for exploration. Widgets require an active Jupyter kernel; the keyframe strip below remains visible in exported/static notebooks.

In [ ]:
FIELD_SELECTED_SAMPLE = 0
FIELD_VIEW_FRAME = 10
FIELD_COMPONENT = "U2"
FIELD_STYLES = {"UT": "continuous", "FT": "points"}
# Close only this cell's prior viewers when rerunning, never other widgets.
for previous in globals().get("viewers", {}).values():
    if previous is not None:
        for widget in previous:
            widget.close()
viewers = {}
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode} · simulation ID {field_diag['sample_ids'][FIELD_SELECTED_SAMPLE]}"))
    plot_field_keyframe_strip(field_diag, sample=FIELD_SELECTED_SAMPLE, component=FIELD_COMPONENT,
                             n_keyframes=5, rows=("truth", "prediction", "error"), plot_style=FIELD_STYLES[mode])
    viewers[mode] = field_sample_viewer(
        field_diag, sample_mode="selected", selected_samples=FIELD_SELECTED_SAMPLE,
        frame=FIELD_VIEW_FRAME, component=FIELD_COMPONENT, plot_style=FIELD_STYLES[mode],
    )

## Sample frame evolution

Follow one specimen's scalar errors through its loading history, complementing the spatial viewer.

In [ ]:
FRAME_EVOLUTION_SAMPLE = FIELD_SELECTED_SAMPLE
for mode, field_diag in fields.items():
    display(Markdown(f"### {mode}"))
    plot_field_sample_frame_evolution(field_diag, sample=FRAME_EVOLUTION_SAMPLE)

## Local motion and field-loss components

New loss trials save signed spatial/temporal errors and specimen-specific large-jump diagnostics. These measure displacement activity, not confirmed strut damage. The plots show both tasks. Old runs report missing artifacts explicitly. Raw loss terms use log axes and are distinct from physical-unit errors.

In [ ]:
from resources.MLmetrics import plot_field_motion_results, plot_field_loss_components, plot_true_field_curve_comparison
plot_field_motion_results(RESULTS_DIR, SPLIT);
plot_field_loss_components(history);
plot_true_field_curve_comparison(RESULTS_DIR, SPLIT);


## Field loss monitoring

The joint objective is shown again for orientation; this section isolates the two field contributions. All axes stay logarithmic.

In [ ]:
plot_dual_loss_history(history, kind="field", best_epoch=BEST_EPOCH, weighted=WEIGHTED_LOSSES);

# Additional joint diagnostics

Four simultaneous comparisons: UT field–curve, FT field–curve, UT–FT curves, and UT–FT fields. Points are joined by unique simulation ID, not assumed row order. High errors in several panels flag jointly difficult specimens; Pearson r is descriptive, not evidence of causation. RMSE has no sign, so use the earlier **bias** diagnostics to distinguish overprediction from underprediction. Native-unit RMSE axes are not interchangeable measures of relative task performance; use baseline skill above for that comparison.

In [ ]:
paired, fig, axes = plot_dual_sample_errors(curves, fields)
display(paired.drop(columns="sample_id").corr().round(3))